<a href="https://colab.research.google.com/github/MElsdk-lab/Biochar_forest_estimation/blob/main/Notebook_1_GEE_Forest_Area_Computation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# ============================================================
# NOTEBOOK 1 — GEE Forest Area Computation
# University of Pittsburgh | Biochar Feedstock Methodology
# ============================================================

In [18]:
# ── CELL 1: Install Libraries ─────────────────────────────────────────────────
!pip install -q earthengine-api geemap

In [19]:
# ── CELL 2: conect to drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_FOLDER = '/content/drive/MyDrive/BiocharProject/'
print('✅ connected to drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ connected to drive


In [20]:
# ── CELL 3: clone repo if not already cloned ─────────────────────
import os
import getpass
import subprocess


if not os.path.exists('/content/Biochar_forest_estimation'):
    PAT = getpass.getpass('Enter PAT: ')
    !git config --global user.email "sdkmajd@gmail.com"
    !git config --global user.name "MElsdk-lab"
    subprocess.run(
        f'git clone https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git',
        shell=True
    )

%cd /content/Biochar_forest_estimation/
!git fetch origin
!git reset --hard origin/main
print('✅ Ready')

/content/Biochar_forest_estimation
From https://github.com/MElsdk-lab/Biochar_forest_estimation
   7cbdf8c..a4fb6f7  main       -> origin/main
HEAD is now at a4fb6f7 update results
✅ Ready


In [21]:
# ── CELL 4: import libraries and data from data_config ─────────────────────

import ee
import geemap
import pandas as pd
import shutil

from data_config import FAO_LSIB_REGION, us_state_names, get_all_countries, build_country_lookup, country_thresholds, state_thresholds
import time

GEE_FOLDER = '/content/drive/MyDrive/GEE_exports/' # Google Drive folder for GEE exports
DATA_FOLDER = '/content/Biochar_forest_estimation/data/' # Local data folder in repo

print('✅ Libraries imported')
print('✅ Data config loaded')

✅ Libraries imported
✅ Data config loaded


In [22]:
# ── CELL 5: Authenticate and initialize Earth Engine & Load Datasets  ─────────────────────
try:
    ee.Initialize(project='spry-blade-487218-n0')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='spry-blade-487218-n0')  # ← add project here too

Hansen_GFC_2024      = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")
GLC_FSC30D_annual    = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/annual')
GLC_FSC30D_five_year = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/five-years-map')

print('✅ Hansen GFC 2024 loaded')
print('✅ GLC FCS30D annual loaded')
print('✅ GLC FCS30D five-year loaded')


✅ Hansen GFC 2024 loaded
✅ GLC FCS30D annual loaded
✅ GLC FCS30D five-year loaded


In [23]:
# ── CELL 6: processing datasets ───────────────────────────────────────

# Select & Mask Hansen Bands
treecover2000 = Hansen_GFC_2024.select('treecover2000')
datamask      = Hansen_GFC_2024.select('datamask')

# Keep pixels with tree cover > 0 and apply Hansen data mask (1 = mapped land)
treecover2000_masked = treecover2000.updateMask(treecover2000.gt(0)) \
                                    .updateMask(datamask.eq(1))

print('✅ treecover2000 masked')

#Load GLC FCS30D Year 2000

glc_2000        = GLC_FSC30D_annual.mosaic().select('b1')
# Filter GLC classes 51-92 (forests) and mask out non-forest pixels
glc_2000_forest = glc_2000.gte(51).And(glc_2000.lte(92)).selfMask()

print('✅ GLC FCS30D 2000 loaded')
print('✅ GLC forest mask created (classes 51-92)')

# Forest Class Definitions
forestClasses = [
    {'code': 51, 'color': '4c7300', 'name': '51 Open evergreen broadleaved'},
    {'code': 52, 'color': '006400', 'name': '52 Closed evergreen broadleaved'},
    {'code': 61, 'color': 'a8c800', 'name': '61 Open deciduous broadleaved'},
    {'code': 62, 'color': '00a000', 'name': '62 Closed deciduous broadleaved'},
    {'code': 71, 'color': '005000', 'name': '71 Open evergreen needleleaved'},
    {'code': 72, 'color': '003c00', 'name': '72 Closed evergreen needleleaved'},
    {'code': 81, 'color': '286400', 'name': '81 Open deciduous needleleaved'},
    {'code': 82, 'color': '285000', 'name': '82 Closed deciduous needleleaved'},
    {'code': 91, 'color': 'a0b432', 'name': '91 Open mixed forest'},
    {'code': 92, 'color': '788200', 'name': '92 Closed mixed forest'},
]

print(f'✅ {len(forestClasses)} forest classes defined')

✅ treecover2000 masked
✅ GLC FCS30D 2000 loaded
✅ GLC forest mask created (classes 51-92)
✅ 10 forest classes defined


In [24]:
def prepare_forest_collection(selected_regions, threshold):
    """
    Prepare a GEE FeatureCollection with forest area per country
    for a given canopy cover threshold.
    Returns a GEE object (not yet computed).
    """
    all_countries = get_all_countries(selected_regions)

    lsib_fao = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017') \
                 .filter(ee.Filter.inList('country_na', all_countries))

    forest_mask = treecover2000_masked.gte(threshold).selfMask().updateMask(datamask.eq(1))
    area_image  = forest_mask.multiply(ee.Image.pixelArea().divide(1e10))

    region_areas = area_image.reduceRegions(
        collection=lsib_fao,
        reducer=ee.Reducer.sum(),
        scale=30,
    )
    region_areas = region_areas.map(lambda f: f.set('threshold', threshold))
    return region_areas


def export_forest_area(selected_regions, thresholds):
    """
    Merge all thresholds into one FeatureCollection and
    submit a single export task to Google Drive.
    """
    combined = ee.FeatureCollection([])

    for threshold in thresholds:
        fc = prepare_forest_collection(selected_regions, threshold)
        combined = combined.merge(fc)

    task = ee.batch.Export.table.toDrive(
        collection=combined,
        description='forest_area_all_thresholds',
        folder='GEE_exports',
        fileNamePrefix='forest_area_all_thresholds',
        fileFormat='CSV',
        selectors=['country_na', 'threshold', 'sum']
    )
    task.start()
    print('✅ Single export task submitted: forest_area_all_thresholds')
    return task


print('✅ prepare_forest_collection() defined')
print('✅ export_forest_area() defined')

✅ prepare_forest_collection() defined
✅ export_forest_area() defined


In [25]:
# ── CELL 8: GEE Functions — US States ────────────────────────────────────────
def prepare_states_forest_collection(selected_states, threshold):
    """
    Prepare a GEE FeatureCollection with forest area per US state
    for a given canopy cover threshold.
    Returns a GEE object (not yet computed).
    """
    tiger_states = ee.FeatureCollection('TIGER/2018/States') \
                     .filter(ee.Filter.inList('NAME', selected_states))

    forest_mask = treecover2000_masked.gte(threshold).selfMask().updateMask(datamask.eq(1))
    area_image  = forest_mask.multiply(ee.Image.pixelArea().divide(1e10))

    states_areas = area_image.reduceRegions(
        collection=tiger_states,
        reducer=ee.Reducer.sum(),
        scale=30,
        maxPixelsPerRegion=1e13
    )
    states_areas = states_areas.map(lambda f: f.set('threshold', threshold))
    return states_areas


def export_states_forest_area(selected_states, thresholds):
    """
    Merge all thresholds into one FeatureCollection and
    submit a single export task to Google Drive.
    """
    combined = ee.FeatureCollection([])

    for threshold in thresholds:
        fc = prepare_states_forest_collection(selected_states, threshold)
        combined = combined.merge(fc)

    task = ee.batch.Export.table.toDrive(
        collection=combined,
        description='states_forest_area_all_thresholds',
        folder='GEE_exports',
        fileNamePrefix='states_forest_area_all_thresholds',
        fileFormat='CSV',
        selectors=['NAME', 'threshold', 'sum']
    )
    task.start()
    print('✅ Single export task submitted: states_forest_area_all_thresholds')
    return task


print('✅ prepare_states_forest_collection() defined')
print('✅ export_states_forest_area() defined')

✅ prepare_states_forest_collection() defined
✅ export_states_forest_area() defined


In [26]:
# ── CELL 9: Run Exports ───────────────────────────────────────────────────────
print('── Submitting country task ──')
country_task = export_forest_area(FAO_LSIB_REGION, country_thresholds)

print('\n── Submitting state task ──')
state_task = export_states_forest_area(us_state_names, state_thresholds)

── Submitting country task ──
✅ Single export task submitted: forest_area_all_thresholds

── Submitting state task ──
✅ Single export task submitted: states_forest_area_all_thresholds


In [ ]:
# ── CELL 10: Monitor Tasks ─────────────────────────────────────────────────────
all_tasks = [country_task, state_task]

for i in range(30):
    print(f'\n── Check {i+1} ──')
    all_done = True
    for task in all_tasks:
        status = task.status()
        print(f"  {status['description']}: {status['state']}")
        if status['state'] not in ['COMPLETED', 'FAILED']:
            all_done = False
    if all_done:
        print('\n✅ All tasks completed!')
        break
    time.sleep(60)



── Check 1 ──
  forest_area_all_thresholds: READY
  states_forest_area_all_thresholds: READY

── Check 2 ──
  forest_area_all_thresholds: READY
  states_forest_area_all_thresholds: READY


In [ ]:
# ── CELL 11: Copy GEE exports from Drive to GitHub repo ──────────────────────

files = [
    'forest_area_all_thresholds.csv',
    'states_forest_area_all_thresholds.csv',
]

for f in files:
    shutil.copy(GEE_FOLDER + f, DATA_FOLDER + f)
    print(f'✅ Copied {f}')

In [ ]:
# ── CELL 12: Push results to GitHub ────────────────────────────────
import subprocess

# ask for PAT only if not already defined
if 'PAT' not in globals(): # Check if PAT is already defined
    import getpass
    PAT = getpass.getpass('Enter PAT: ') # Prompt user for PAT

%cd /content/Biochar_forest_estimation/
!git add . # Stage all changes
!git commit -m "update results" # Commit changes

result = subprocess.run( # Run git push command
    f'git push https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git main',
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0: # Check if push was successful
    print('✅ Pushed to GitHub')
else:
    print('❌ Push failed')
    print(result.stderr)